# AUROC Estimate For Interpretation Models

This notebook reproduces the feature engineering used in `250607_Interpretation.ipynb` and estimates AUROC in a separate workflow.

- It does **not** overwrite any existing interpretation outputs.
- It uses the same model choice as the interpretation notebook: `XGBClassifier` for the predelivery predictor and `flaml.default.XGBClassifier` for the early- and late-death predictors.
- Missing values are imputed with the **mean** inside each training fold to avoid leakage.
- AUROC is estimated with stratified cross-validation.
- It also compares the baseline feature set with a version that replaces raw `Birthweight` and `Gestation` by `Weight percentile = Birthweight / Gestation` when both variables are available.


In [5]:
import pandas as pd
import numpy as np

from flaml.default import XGBClassifier as ZeroShotXGBClassifier
from xgboost import XGBClassifier

from sklearn.base import clone
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold


dataset_names = ["ml_d1_predelivery", "ml_d2_earlydeath", "ml_d3_latedeath"]


def prepare_interpretation_dataframe(dataset_name, add_weight_percentile=False):
    df = pd.read_csv(f"../data/processed/{dataset_name}.csv")
    target_col = df.columns[0]

    if "_d1_" in dataset_name:
        target = [None if str(v) == "nan" else int(v == "Miscarriage or Stillbirth") for v in df[target_col].tolist()]
    else:
        target = [None if str(v) == "nan" else int(v == "Dead") for v in df[target_col].tolist()]

    df[target_col] = pd.Series(target, dtype="float")
    df = df.rename(columns={target_col: "Death"})

    if "Birthweight Measure" in df.columns:
        df = df.drop(columns=["Birthweight Measure"], errors="ignore")
    if "Type of Delivery Place" in df.columns:
        df = df.drop(columns=["Type of Delivery Place"], errors="ignore")

    bmi = np.array(df["Maternal Weight"].tolist()) / (np.array(df["Maternal Height"].tolist()) / 100) ** 2
    df["BMI"] = np.clip(bmi, 10, 50)
    df = df.drop(columns=["Maternal Weight", "Maternal Height"], errors="ignore")

    school_levels = []
    for x in df["School Level"].tolist():
        x = str(x)
        if x == "nan":
            school_levels.append(None)
        else:
            school_levels.append(int(x[0]) - 1)
    df["School Level"] = pd.Series(school_levels, dtype="float")

    if "Pre-term Delivery" in df.columns:
        preterm = []
        for x in df["Pre-term Delivery"].tolist():
            x = str(x)
            if x == "nan":
                preterm.append(None)
            else:
                preterm.append(int(x == "Preterm"))
        df["Pre-term Delivery"] = pd.Series(preterm, dtype="float")

    if "Mode of Delivery" in df.columns:
        cesarean = []
        for x in df["Mode of Delivery"].tolist():
            x = str(x)
            if x == "nan":
                cesarean.append(None)
            else:
                cesarean.append(int(x == "Cesarean"))
        df["Mode of Delivery Cesarea"] = pd.Series(cesarean, dtype="float")
        df = df.drop(columns=["Mode of Delivery"], errors="ignore")

    if "Baby Sex" in df.columns:
        baby_sex_female = []
        for x in df["Baby Sex"].tolist():
            x = str(x)
            if x == "nan":
                baby_sex_female.append(None)
            else:
                baby_sex_female.append(int(x == "Female"))
        df["Baby Sex Female"] = pd.Series(baby_sex_female, dtype="float")
        df = df.drop(columns=["Baby Sex"], errors="ignore")

    if "Delivery Place" in df.columns:
        delivery_home = []
        for x in df["Delivery Place"].tolist():
            x = str(x)
            if x == "nan":
                delivery_home.append(None)
            else:
                delivery_home.append(int(x in ["Home", "Other"]))
        df["Delivery Place Home"] = pd.Series(delivery_home, dtype="float")
        df = df.drop(columns=["Delivery Place"], errors="ignore")

    if add_weight_percentile and {"Birthweight", "Gestation"}.issubset(df.columns):
        gestation = pd.to_numeric(df["Gestation"], errors="coerce")
        birthweight = pd.to_numeric(df["Birthweight"], errors="coerce")
        valid = gestation > 0
        weight_percentile = pd.Series(np.nan, index=df.index, dtype="float")
        weight_percentile.loc[valid] = birthweight.loc[valid] / gestation.loc[valid]
        df["Weight percentile"] = weight_percentile
        df = df.drop(columns=["Birthweight", "Gestation"], errors="ignore")

    df = df[df["Death"].notnull()].copy()
    df["Death"] = df["Death"].astype(int)
    return df


def build_model(model_class):
    model = model_class()
    params = model.get_params()
    updates = {}

    if "random_state" in params:
        updates["random_state"] = 42
    if "seed" in params:
        updates["seed"] = 42
    if "n_jobs" in params:
        updates["n_jobs"] = -1
    if "eval_metric" in params:
        updates["eval_metric"] = "auc"
    if "use_label_encoder" in params:
        updates["use_label_encoder"] = False

    if updates:
        model.set_params(**updates)
    return model


def predict_positive_score(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    if hasattr(model, "decision_function"):
        return model.decision_function(X)
    raise ValueError("Model does not expose predict_proba or decision_function.")


def make_stratified_splits(df, n_splits=5):
    X = df.iloc[:, 1:].copy()
    y = df.iloc[:, 0].astype(int).copy()

    positive_count = int(y.sum())
    negative_count = int((1 - y).sum())
    n_splits = min(n_splits, positive_count, negative_count)
    if n_splits < 2:
        raise ValueError("Not enough positive and negative examples to estimate AUROC with stratified CV.")

    splitter = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    splits = list(splitter.split(X, y))
    return X, y, splits


def summarize_fold_scores(df, X, y, fold_scores):
    fold_scores = pd.DataFrame(fold_scores)
    summary = {
        "n_samples": len(df),
        "n_features": X.shape[1],
        "n_positive": int(y.sum()),
        "n_negative": int((1 - y).sum()),
        "n_splits": len(fold_scores),
        "mean_auroc": fold_scores["auroc"].mean(),
        "std_auroc": fold_scores["auroc"].std(ddof=1) if len(fold_scores) > 1 else 0.0,
        "min_auroc": fold_scores["auroc"].min(),
        "max_auroc": fold_scores["auroc"].max(),
    }
    return fold_scores, summary


def estimate_auroc(model_class, df, n_splits=5):
    X, y, splits = make_stratified_splits(df, n_splits=n_splits)
    fold_scores = []

    for fold_idx, (train_idx, test_idx) in enumerate(splits, start=1):
        X_train = X.iloc[train_idx]
        X_test = X.iloc[test_idx]
        y_train = y.iloc[train_idx]
        y_test = y.iloc[test_idx]

        imputer = SimpleImputer(strategy="mean")
        X_train_imp = pd.DataFrame(imputer.fit_transform(X_train), columns=X.columns, index=X_train.index)
        X_test_imp = pd.DataFrame(imputer.transform(X_test), columns=X.columns, index=X_test.index)

        model = build_model(model_class)
        model.fit(X_train_imp, y_train)
        y_score = predict_positive_score(model, X_test_imp)
        fold_auroc = roc_auc_score(y_test, y_score)
        fold_scores.append({"fold": fold_idx, "auroc": fold_auroc})

    return summarize_fold_scores(df, X, y, fold_scores)


def estimate_optimism_corrected_auroc(model_class, df, n_bootstrap=200, random_state=42):
    """Bootstrap optimism correction (Harrell's method) for AUROC."""
    rng = np.random.default_rng(random_state)

    X_full = df.iloc[:, 1:].copy()
    y_full = df.iloc[:, 0].astype(int).copy()
    n = len(df)

    # Impute full dataset once for the apparent model and test-on-original evaluations
    imputer_full = SimpleImputer(strategy="mean")
    X_full_imp = pd.DataFrame(imputer_full.fit_transform(X_full), columns=X_full.columns)
    y_full_arr = y_full.values

    # Apparent AUC: fit and evaluate on the full dataset
    model_apparent = build_model(model_class)
    model_apparent.fit(X_full_imp, y_full_arr)
    apparent_score = predict_positive_score(model_apparent, X_full_imp)
    apparent_auc = roc_auc_score(y_full_arr, apparent_score)

    optimisms = []
    for i in range(n_bootstrap):
        idx = rng.integers(0, n, size=n)
        X_boot = X_full.iloc[idx]
        y_boot = y_full.iloc[idx].values

        # Fit imputer on bootstrap sample only (avoid leakage)
        imputer_boot = SimpleImputer(strategy="mean")
        X_boot_imp = pd.DataFrame(imputer_boot.fit_transform(X_boot), columns=X_full.columns)

        model_boot = build_model(model_class)
        model_boot.fit(X_boot_imp, y_boot)

        # AUC on bootstrap sample itself
        boot_score_on_boot = predict_positive_score(model_boot, X_boot_imp)
        auc_on_boot = roc_auc_score(y_boot, boot_score_on_boot)

        # AUC on original dataset (apply bootstrap imputer to original features)
        X_orig_imp = pd.DataFrame(imputer_boot.transform(X_full), columns=X_full.columns)
        boot_score_on_orig = predict_positive_score(model_boot, X_orig_imp)
        auc_on_orig = roc_auc_score(y_full_arr, boot_score_on_orig)

        optimisms.append(auc_on_boot - auc_on_orig)

    mean_optimism = float(np.mean(optimisms))
    corrected_auc = apparent_auc - mean_optimism

    return {
        "apparent_auc": apparent_auc,
        "mean_optimism": mean_optimism,
        "corrected_auc": corrected_auc,
        "std_optimism": float(np.std(optimisms, ddof=1)),
        "n_bootstrap": n_bootstrap,
        "n_samples": n,
        "n_features": X_full.shape[1],
        "n_positive": int(y_full_arr.sum()),
        "n_negative": int((1 - y_full_arr).sum()),
    }


def build_tabpfn_model():
    from tabpfn_client import TabPFNClassifier
    return TabPFNClassifier(ignore_pretraining_limits=True)

In [2]:
all_fold_results = []
all_summaries = []
comparison_rows = []

for dataset_name in dataset_names:
    model_class = XGBClassifier if "_d1_" in dataset_name else ZeroShotXGBClassifier

    baseline_df = prepare_interpretation_dataframe(dataset_name, add_weight_percentile=False)
    baseline_fold_results, baseline_summary = estimate_auroc(model_class, baseline_df, n_splits=5)
    baseline_fold_results.insert(0, "model_variant", "baseline")
    baseline_fold_results.insert(0, "dataset_name", dataset_name)
    all_fold_results.append(baseline_fold_results)
    all_summaries.append({"dataset_name": dataset_name, "model_variant": "baseline", **baseline_summary})

    augmented_summary = None
    if {"Birthweight", "Gestation"}.issubset(baseline_df.columns):
        augmented_df = prepare_interpretation_dataframe(dataset_name, add_weight_percentile=True)
        augmented_fold_results, augmented_summary = estimate_auroc(model_class, augmented_df, n_splits=5)
        augmented_fold_results.insert(0, "model_variant", "with_weight_percentile")
        augmented_fold_results.insert(0, "dataset_name", dataset_name)
        all_fold_results.append(augmented_fold_results)
        all_summaries.append({"dataset_name": dataset_name, "model_variant": "with_weight_percentile", **augmented_summary})

    comparison_rows.append({
        "dataset_name": dataset_name,
        "baseline_mean_auroc": baseline_summary["mean_auroc"],
        "weight_percentile_mean_auroc": None if augmented_summary is None else augmented_summary["mean_auroc"],
        "delta_mean_auroc": None if augmented_summary is None else augmented_summary["mean_auroc"] - baseline_summary["mean_auroc"],
        "note": "Weight percentile unavailable because Birthweight is not present." if augmented_summary is None else "Replaced Birthweight and Gestation with Weight percentile = Birthweight / Gestation.",
    })

fold_results_df = pd.concat(all_fold_results, ignore_index=True)
summary_df = pd.DataFrame(all_summaries)
comparison_df = pd.DataFrame(comparison_rows)

display(comparison_df)
display(summary_df)
display(fold_results_df)


,dataset_name,baseline_mean_auroc,weight_percentile_mean_auroc,delta_mean_auroc,note
0,ml_d1_predelivery,0.621536,NaN,NaN,Weight percentile unavailable because Birthwei...
1,ml_d2_earlydeath,0.786107,0.798665,0.012558,Replaced Birthweight and Gestation with Weight...
2,ml_d3_latedeath,0.610922,0.633549,0.022627,Replaced Birthweight and Gestation with Weight...


,dataset_name,model_variant,n_samples,n_features,n_positive,n_negative,n_splits,mean_auroc,std_auroc,min_auroc,max_auroc
0,ml_d1_predelivery,baseline,10596,11,174,10422,5,0.621536,0.038385,0.581903,0.670038
1,ml_d2_earlydeath,baseline,10409,19,112,10297,5,0.786107,0.050312,0.741897,0.861805
2,ml_d2_earlydeath,with_weight_percentile,10409,18,112,10297,5,0.798665,0.045626,0.746542,0.848919
3,ml_d3_latedeath,baseline,10288,19,22,10266,5,0.610922,0.053572,0.546322,0.680988
4,ml_d3_latedeath,with_weight_percentile,10288,18,22,10266,5,0.633549,0.023982,0.617755,0.675110


,dataset_name,model_variant,fold,auroc
0,ml_d1_predelivery,baseline,1,0.670038
1,ml_d1_predelivery,baseline,2,0.653752
2,ml_d1_predelivery,baseline,3,0.607650
3,ml_d1_predelivery,baseline,4,0.581903
4,ml_d1_predelivery,baseline,5,0.594338
5,ml_d2_earlydeath,baseline,1,0.811408
6,ml_d2_earlydeath,baseline,2,0.861805
7,ml_d2_earlydeath,baseline,3,0.748020
8,ml_d2_earlydeath,baseline,4,0.741897
9,ml_d2_earlydeath,baseline,5,0.767407


## Baseline AUROC Comparison: XGBoost vs TabPFN

This comparison uses the baseline interpretation feature set only, without `Weight percentile`.
Missing values are mean-imputed within each training fold for both model families.


In [ ]:
from tabpfn_client import TabPFNClassifier


def build_tabpfn_model():
    return TabPFNClassifier(ignore_pretraining_limits=True)


baseline_model_comparison_rows = []
baseline_model_fold_rows = []

for dataset_name in dataset_names:
    print(f"\nProcessing baseline AUROC comparison for {dataset_name}...")
    baseline_df = prepare_interpretation_dataframe(dataset_name, add_weight_percentile=False)
    xgb_model_class = XGBClassifier if "_d1_" in dataset_name else ZeroShotXGBClassifier
    X, y, splits = make_stratified_splits(baseline_df, n_splits=5)
    print(f"  Using {len(splits)} shared stratified folds for both model families.")

    xgb_fold_scores = []
    tabpfn_fold_scores = []

    for fold_idx, (train_idx, test_idx) in enumerate(splits, start=1):
        print(f"  Shared fold {fold_idx}/{len(splits)}")
        X_train = X.iloc[train_idx]
        X_test = X.iloc[test_idx]
        y_train = y.iloc[train_idx]
        y_test = y.iloc[test_idx]

        imputer = SimpleImputer(strategy="mean")
        X_train_imp = pd.DataFrame(imputer.fit_transform(X_train), columns=X.columns, index=X_train.index)
        X_test_imp = pd.DataFrame(imputer.transform(X_test), columns=X.columns, index=X_test.index)

        print(f"    fitting XGBoost-style model on fold {fold_idx}")
        xgb_model = build_model(xgb_model_class)
        xgb_model.fit(X_train_imp, y_train)
        xgb_score = predict_positive_score(xgb_model, X_test_imp)
        xgb_auroc = roc_auc_score(y_test, xgb_score)
        xgb_fold_scores.append({"fold": fold_idx, "auroc": xgb_auroc})
        print(f"    fold {fold_idx} XGBoost AUROC: {xgb_auroc:.4f}")

        print(f"    fitting TabPFN on fold {fold_idx}")
        tabpfn_model = build_tabpfn_model()
        tabpfn_model.fit(X_train_imp, y_train)
        print(f"    predicting with TabPFN on fold {fold_idx}")
        tabpfn_score = tabpfn_model.predict_proba(X_test_imp)[:, 1]
        tabpfn_auroc = roc_auc_score(y_test, tabpfn_score)
        tabpfn_fold_scores.append({"fold": fold_idx, "auroc": tabpfn_auroc})
        print(f"    fold {fold_idx} TabPFN AUROC: {tabpfn_auroc:.4f}")
        print(f"    fold {fold_idx} delta TabPFN - XGBoost: {tabpfn_auroc - xgb_auroc:.4f}")

    xgb_fold_results, xgb_summary = summarize_fold_scores(baseline_df, X, y, xgb_fold_scores)
    tabpfn_fold_results, tabpfn_summary = summarize_fold_scores(baseline_df, X, y, tabpfn_fold_scores)
    print(f"  Finished XGBoost-style model for {dataset_name}. Mean AUROC: {xgb_summary['mean_auroc']:.4f}")
    xgb_fold_results.insert(0, "model_family", "xgboost")
    xgb_fold_results.insert(0, "dataset_name", dataset_name)
    baseline_model_fold_rows.append(xgb_fold_results)

    print(f"  Finished TabPFN for {dataset_name}. Mean AUROC: {tabpfn_summary['mean_auroc']:.4f}")
    tabpfn_fold_results.insert(0, "model_family", "tabpfn")
    tabpfn_fold_results.insert(0, "dataset_name", dataset_name)
    baseline_model_fold_rows.append(tabpfn_fold_results)

    baseline_model_comparison_rows.append({
        "dataset_name": dataset_name,
        "xgboost_mean_auroc": xgb_summary["mean_auroc"],
        "tabpfn_mean_auroc": tabpfn_summary["mean_auroc"],
        "delta_tabpfn_minus_xgboost": tabpfn_summary["mean_auroc"] - xgb_summary["mean_auroc"],
        "xgboost_std_auroc": xgb_summary["std_auroc"],
        "tabpfn_std_auroc": tabpfn_summary["std_auroc"],
    })
    print(f"Completed comparison for {dataset_name}. Delta TabPFN - XGBoost: {tabpfn_summary['mean_auroc'] - xgb_summary['mean_auroc']:.4f}")

baseline_model_comparison_df = pd.DataFrame(baseline_model_comparison_rows)
baseline_model_fold_df = pd.concat(baseline_model_fold_rows, ignore_index=True)

print("\nBaseline XGBoost vs TabPFN comparison complete.")
display(baseline_model_comparison_df)
display(baseline_model_fold_df)

## Optimism-Corrected AUC: Bootstrap Validation (XGBoost)

Estimates bias-corrected AUROC using Harrell's bootstrap optimism correction (200 bootstrap iterations).
Missing values are mean-imputed from the bootstrap sample (for the bootstrap model) and from the full dataset (for the apparent model), to avoid leakage.

In [6]:
bootstrap_rows = []

for dataset_name in dataset_names:
    print(f"\nBootstrap optimism correction for {dataset_name}...")
    baseline_df = prepare_interpretation_dataframe(dataset_name, add_weight_percentile=False)
    model_class = XGBClassifier if "_d1_" in dataset_name else ZeroShotXGBClassifier

    result = estimate_optimism_corrected_auroc(model_class, baseline_df, n_bootstrap=200)
    bootstrap_rows.append({"dataset_name": dataset_name, **result})
    print(f"  Apparent AUC:   {result['apparent_auc']:.4f}")
    print(f"  Mean optimism:  {result['mean_optimism']:.4f}  (std: {result['std_optimism']:.4f})")
    print(f"  Corrected AUC:  {result['corrected_auc']:.4f}")

bootstrap_df = pd.DataFrame(bootstrap_rows)
display(bootstrap_df)


Bootstrap optimism correction for ml_d1_predelivery...
  Apparent AUC:   0.9989
  Mean optimism:  0.1285  (std: 0.0186)
  Corrected AUC:  0.8704

Bootstrap optimism correction for ml_d2_earlydeath...
  Apparent AUC:   1.0000
  Mean optimism:  0.0745  (std: 0.0153)
  Corrected AUC:  0.9255

Bootstrap optimism correction for ml_d3_latedeath...
  Apparent AUC:   0.9999
  Mean optimism:  0.1254  (std: 0.0470)
  Corrected AUC:  0.8745


,dataset_name,apparent_auc,mean_optimism,corrected_auc,std_optimism,n_bootstrap,n_samples,n_features,n_positive,n_negative
0,ml_d1_predelivery,0.998921,0.128481,0.870440,0.018567,200,10596,11,174,10422
1,ml_d2_earlydeath,0.999954,0.074502,0.925452,0.015324,200,10409,19,112,10297
2,ml_d3_latedeath,0.999938,0.125392,0.874546,0.047009,200,10288,19,22,10266
